# KAF-1 — Partitioning & hot partitions

**Break → Detect → Fix → Prove.** A Kafka topic's parallelism is its **partitions**. The producer
picks a partition by **hashing the message key**, so a bad key (one dominant value) sends most
messages to **one partition** — a *hot partition*. That partition's consumer falls behind while
the others idle, and ordering is only guaranteed **within** a partition.

**Pre-requisite:** `make up` (Kafka at `localhost:29092` for producers, `kafka:9092` for Spark) —
inspect topics live in **kafka-ui** at http://localhost:8080. **Laptop-safe:** bounded produce,
batch reads, topics deleted at teardown.

In [ ]:
from common.spark_session import spark, display_df
from common.kafka_helpers import ensure_topic, produce_events, topic_end_offsets, delete_topic, SPARK_BOOTSTRAP
from pyspark.sql import functions as F

TOPIC = "kaf1_orders"
PARTITIONS = 3
print("Spark reads via", SPARK_BOOTSTRAP, "| kafka-ui:", "http://localhost:8080")

## Step 0 — a topic with 3 partitions

In [ ]:
ensure_topic(TOPIC, num_partitions=PARTITIONS, recreate=True)
print(f"created {TOPIC} with {PARTITIONS} partitions")

## 1. Break it — a dominant key

A realistic mistake: keying by something lopsided (e.g. `country` where 90% of orders are one
country, or a `null`/constant key). Here 90% of events share the key `"HOT"`.

In [ ]:
produce_events(TOPIC, 3000, key_fn=lambda i: "HOT" if i % 10 else f"cust-{i}")
skewed = topic_end_offsets(TOPIC)
print("messages per partition:", skewed)
print(f"hottest partition holds {max(skewed.values())} / {sum(skewed.values())} "
      f"({100*max(skewed.values())/sum(skewed.values()):.0f}%)")

## 2. Detect it

Two views of the same skew:
- **kafka-ui** (http://localhost:8080 → topic `kaf1_orders` → Partitions) shows one partition's
  offset far ahead of the others.
- In **Spark**, the Kafka source exposes a `partition` column — grouping by it shows one partition
  carrying the load (its consumer task does most of the work, and lags).

In [ ]:
by_part = (spark.read.format("kafka")
           .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
           .option("subscribe", TOPIC).option("startingOffsets", "earliest").load()
           .groupBy("partition").count().orderBy("partition"))
by_part.show()

## 3. Diagnose

`partition = hash(key) % num_partitions`. A dominant key collapses to one partition, so:
- that partition's consumer lags while others idle (wasted parallelism),
- you can't fix it by adding partitions (the hot key still hashes to one),
- ordering holds only *within* a partition (rarely the global order you wanted).

## 4. Fix it — a high-cardinality key (or salt the hot one)

Key by something that spreads evenly (here a per-customer id). If one key is legitimately hot,
**salt** it (`key + "-" + rand(0..N)`) and merge downstream — same idea as SPK-1 salting.

In [ ]:
ensure_topic(TOPIC, num_partitions=PARTITIONS, recreate=True)   # fresh topic
produce_events(TOPIC, 3000, key_fn=lambda i: f"cust-{i % 300}")  # 300 distinct keys -> even spread
balanced = topic_end_offsets(TOPIC)
print("messages per partition:", balanced)

## 5. Prove it

In [ ]:
def spread(d):
    lo, hi = min(d.values()), max(d.values())
    return f"min={lo} max={hi} skew(max/min)={hi/max(lo,1):.1f}x"
print("dominant key :", skewed,   "->", spread(skewed))
print("balanced key :", balanced, "->", spread(balanced))
print("\nEven partitions = even consumer load + full parallelism.")

## Takeaways & "in real production…"

- **Partition key = parallelism + ordering.** Choose a high-cardinality key aligned to how you
  consume; avoid `null`/constant/lopsided keys.
- A hot key can't be fixed by adding partitions — **salt** it (and merge downstream) or rekey.
- Watch **per-partition consumer lag** (kafka-ui / `consumer_group_lag`, see KAF-2); a single
  lagging partition is the hot-partition signature.
- Ordering is per-partition only — design for that.

## Teardown

In [ ]:
delete_topic(TOPIC)
print("Deleted", TOPIC, "(`make clean` also clears any local state).")